# Pediatric Wrist Fracture Detection Using Faster R-CNN (GRAZPEDWRI-DX)
**End-to-End Training, Evaluation & Ablation Notebook on Google Colab (Tesla T4 GPU)**

This notebook connects your VS Code codebase with Google Colab's GPU acceleration.

## 1. Mount Google Drive & Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

# Mount Google Drive where dataset zips and checkpoints are stored
from google.colab import drive
drive.mount('/content/drive')

## 2. Sync Repository & Install Requirements

In [ ]:
%cd /content
# If using GitHub repo sync:
# !git clone https://github.com/<your-username>/GRAZPEDWRI_Faster_RCNN.git
# %cd GRAZPEDWRI_Faster_RCNN

# Install project dependencies
!pip install -r requirements.txt

## 3. Extract Dataset to Fast Local NVMe Storage (`/content/dataset`)

In [ ]:
import os
import zipfile

os.makedirs('/content/dataset/images', exist_ok=True)
os.makedirs('/content/dataset/labels', exist_ok=True)

# Adjust path to your Google Drive location
DRIVE_DATASET_DIR = '/content/drive/MyDrive/GRAZPEDWRI_Dataset'

# Extract image parts
for part in ['images_part1.zip', 'images_part2.zip', 'images_part3.zip', 'images_part4.zip']:
    zip_path = os.path.join(DRIVE_DATASET_DIR, part)
    if os.path.exists(zip_path):
        print(f'Extracting {part}...')
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/dataset/images')

# Extract dedicated fracture YOLO annotations
labels_zip = os.path.join(DRIVE_DATASET_DIR, 'fracture_labels_yolo.zip')
if os.path.exists(labels_zip):
    print('Extracting fracture labels...')
    with zipfile.ZipFile(labels_zip, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset/labels')

print('Extraction complete.')

## 4. Phase 1: Audit Dataset & Verify Patient-Level Split

In [ ]:
!python scripts/inspect_dataset.py --config configs/baseline.yaml --data-root /content/dataset

## 5. Phase 2 & 3: Run Automated Unit Tests

In [ ]:
!pytest tests/ -v

## 6. Phase 4: Baseline Model Training

In [ ]:
# Launch TensorBoard inside Colab
%load_ext tensorboard
%tensorboard --logdir outputs/logs

# Run baseline training with mixed precision
!python scripts/train.py --config configs/baseline.yaml --data-root /content/dataset

## 7. Phase 5: Model Evaluation on Validation & Test Sets

In [ ]:
!python scripts/evaluate.py --config configs/baseline.yaml --checkpoint checkpoints/best_model.pth --split val
# Final evaluation on locked test set:
# !python scripts/evaluate.py --config configs/baseline.yaml --checkpoint checkpoints/best_model.pth --split test

## 8. Phase 6: Controlled Ablation Studies (EXP-01 through EXP-05)
Execute specific ablation experiments (Medical Augmentation, Alternative Backbones, Resolution, Optimizers) and inspect the quantitative summary table.

In [ ]:
# Run Medical Augmentation experiment (EXP-02)
!python scripts/run_experiment.py --experiment exp_02_augmentation --data-root /content/dataset

# Run Alternative Backbone ablation (EXP-03: ResNet-101 vs MobileNetV3)
# !python scripts/run_experiment.py --experiment exp_03_backbone --variant resnet101_fpn --data-root /content/dataset
# !python scripts/run_experiment.py --experiment exp_03_backbone --variant mobilenet_v3_large_fpn --data-root /content/dataset

# Display benchmark summary table
import pandas as pd
df = pd.read_csv('outputs/metrics/experiments_summary.csv')
display(df)

## 9. Phase 7: Inference & Visual Detection

In [ ]:
!python scripts/predict.py --image /content/dataset/images/image_001.png --checkpoint checkpoints/best_model.pth --threshold 0.5